# Lab 06-08 · Thévenin and Norton Equivalents

Companion to chapter [06-08](../../chapters/06-08-thevenin-and-norton.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import shutil
import subprocess
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)

def run_ngspice(path):
    out = subprocess.run(["ngspice", "-b", path], capture_output=True, text=True).stdout
    vals = {}
    for line in out.splitlines():
        parts = line.split("=")
        if len(parts) == 2:
            try:
                vals[parts[0].strip()] = float(parts[1])
            except ValueError:
                pass
    return vals

ngspice available: True


## Part 1 — Three methods, one answer

[thevenin.cir](thevenin.cir) finds each equivalent by open circuit, short circuit and test source.

**🔮 Predict:** which method cannot be used for Example 06-08.3, and why?

In [2]:
if HAVE_NGSPICE:
    ng = run_ngspice("thevenin.cir")
    print(f"Example 06-08.1: v_oc = {ng['ex1voc']:.4f} V, i_sc = {ng['ex1isc'] * 1e3:.4f} mA, "
          f"v_oc/i_sc = {ng['ex1voc'] / ng['ex1isc']:.1f} ohm, test source = {ng['ex1rth']:.1f} ohm")
    print(f"Example 06-08.2: V_th = {ng['ex2voc'] * 1e3:.3f} mV, R_th = {ng['ex2rth']:.2f} ohm")
    print(f"Example 06-08.3: V_th = {ng['ex3voc']:.4f} V, R_th (test source) = {ng['ex3rth']:.2f} ohm")
    assert abs(ng["ex1voc"] / ng["ex1isc"] - ng["ex1rth"]) < 1e-5 * ng["ex1rth"]   # ngspice prints 6 digits
    assert abs(ng["ex3rth"] - 1e4 / 11) < 0.01
else:
    print("ngspice not found; see SETUP.md. Skipping.")
# Method 2 (switch off sources, combine resistors) for Example 06-08.1:
R_th_m2 = 6e3 * 3e3 / (6e3 + 3e3) + 1e3
print(f"Method 2 for Example 06-08.1: {R_th_m2:.0f} ohm")

Example 06-08.1: v_oc = 4.0000 V, i_sc = 1.3333 mA, v_oc/i_sc = 3000.0 ohm, test source = 3000.0 ohm
Example 06-08.2: V_th = 119.048 mV, R_th = 1023.81 ohm
Example 06-08.3: V_th = -4.0000 V, R_th (test source) = 909.09 ohm
Method 2 for Example 06-08.1: 3000 ohm


Method 2 (switch off the independent sources and combine resistors) fails for Example 06-08.3, because the VCVS is not a resistor and cannot be combined in series or parallel. The open/short-circuit ratio and the test source both work.

## Part 2 — Equivalent at the terminals, not inside

Sweep the load on Example 06-08.1's original circuit and on its Thévenin and Norton equivalents.

**🔮 Predict:** the load voltages will overlap. Will the power dissipated *inside* the three circuits also overlap?

In [3]:
def original(RL):                     # 12 V, 6k top, 3k bottom, 1k series; returns v_L and internal power (V, W)
    Rp = 1e3 + RL
    Rlow = 3e3 * Rp / (3e3 + Rp)
    vm = 12 * Rlow / (6e3 + Rlow)
    vL = vm * RL / Rp
    i_src = (12 - vm) / 6e3
    return vL, 12 * i_src - vL ** 2 / RL

def thevenin(RL, V=4.0, R=3e3):
    i = V / (R + RL)
    return i * RL, i ** 2 * R

def norton(RL, I=4 / 3e3, R=3e3):
    vL = I * R * RL / (R + RL)
    return vL, vL ** 2 / R

RL = np.logspace(1, 6, 200)
cur = {name: np.array([f(r) for r in RL]) for name, f in (("original", original), ("Thevenin", thevenin), ("Norton", norton))}
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.2))
for name, arr in cur.items():
    axes[0].semilogx(RL, arr[:, 0], label=name)
    axes[1].loglog(RL, arr[:, 1] * 1e3, label=name)
axes[0].set_xlabel("load (Ω)"); axes[0].set_ylabel("load voltage (V)"); axes[0].legend()
axes[1].set_xlabel("load (Ω)"); axes[1].set_ylabel("internal power (mW)"); axes[1].legend()
plt.tight_layout(); plt.show()
assert np.allclose(cur["original"][:, 0], cur["Thevenin"][:, 0]) and np.allclose(cur["original"][:, 0], cur["Norton"][:, 0])
for name, f in (("original", original), ("Thevenin", thevenin), ("Norton", norton)):
    print(f"{name:9s} at 1 kOhm load: v_L = {f(1e3)[0]:.4f} V, internal power = {f(1e3)[1] * 1e3:.4f} mW")

original  at 1 kOhm load: v_L = 1.0000 V, internal power = 19.0000 mW
Thevenin  at 1 kOhm load: v_L = 1.0000 V, internal power = 3.0000 mW
Norton    at 1 kOhm load: v_L = 1.0000 V, internal power = 0.3333 mW


The load voltages coincide exactly; the internal powers do not. At a 1 kΩ load the original circuit dissipates 19 mW inside, its Thévenin equivalent 3 mW and its Norton equivalent 0.333 mW. Efficiency cannot be read from an equivalent.

## Part 3 — A PoE device as a constant-power load

The Type 4 port of Example 06-08.4 (52 V behind 6.25 Ω) feeds a device that draws a fixed power $P$.

**🔮 Predict:** above what power does no operating point exist, and what is the device voltage just before that?

In [4]:
V, R = 52.0, 6.25
P = np.linspace(1, 108, 400)
disc = V ** 2 - 4 * P * R
v_hi = (V + np.sqrt(disc)) / 2
v_lo = (V - np.sqrt(disc)) / 2
P_max = V ** 2 / (4 * R)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(P, v_hi, label="normal operating point")
ax.plot(P, v_lo, "--", label="second (unstable) solution")
ax.axvline(71.3, color="0.5", lw=0.8); ax.text(72, 5, "Class 8 PD", fontsize=8)
ax.set_xlabel("device power (W)"); ax.set_ylabel("device voltage (V)"); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()
v100 = (V + np.sqrt(V ** 2 - 4 * 100 * R)) / 2
print(f"P_max = {P_max:.2f} W at {V / 2:.0f} V; at 100 W the device would sit at {v100:.2f} V")
assert abs(P_max - 108.16) < 1e-9

P_max = 108.16 W at 26 V; at 100 W the device would sit at 33.14 V


Real solutions exist only up to $V^2/4R$ = 108.2 W, where the device voltage has fallen to half the source voltage, 26 V. Long before that, the device's own minimum input voltage would stop it: the class budget, not the Thévenin limit, sets the practical maximum. (Which minimum applies is not settled here: the standard's tables list about 43 V at the device for Class 8, while this simple cable model puts a 71.3 W device at 41.18 V; the chapter and research-log entry R-105 record that unresolved discrepancy.)

## Part 4 — Two-load measurements with noise

Repeat Example 06-08.5's measurement with 0.1% random reading error, choosing the second load from 1 MΩ to 100 GΩ.

**🔮 Predict:** which second load gives the most reliable estimate of the 200 MΩ source resistance?

In [5]:
Vth, Rth = 0.300, 200.0                          # V, MOhm
rng = np.random.default_rng(608)
R1 = 10.0
R2s = np.logspace(0, 5, 21)
R2s = R2s[np.abs(R2s - R1) / R1 > 0.05]          # a second load equal to the first gives no information
spread = []
for R2 in R2s:
    est = []
    for _ in range(400):
        v1 = Vth * R1 / (Rth + R1) * (1 + 1e-3 * rng.standard_normal())
        v2 = Vth * R2 / (Rth + R2) * (1 + 1e-3 * rng.standard_normal())
        a, b = np.linalg.solve([[1, 1 / R1], [1, 1 / R2]], [1 / v1, 1 / v2])
        est.append(b / a)
    spread.append(np.std(est) / Rth)
spread = np.array(spread)
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.loglog(R2s, spread * 100, "o-")
ax.axvline(Rth, color="0.5", ls="--", lw=0.8); ax.text(Rth * 1.2, spread.max() * 50, "R_th", fontsize=8)
ax.set_xlabel("second load (MΩ)"); ax.set_ylabel("relative spread of R_th estimate (%)"); ax.grid(True, which="both", alpha=0.3)
plt.show()
best = R2s[np.argmin(spread)]
print(f"most reliable second load: about {best:.0f} MOhm (spread {spread.min() * 100:.3f} %)")
assert best > 100

most reliable second load: about 10000 MOhm (spread 0.141 %)


Loads close to the first one (10 MΩ) give nearly the same reading twice, and the estimate drowns in noise. The best second load is one that moves the reading far from the first, at or well above $R_{th}$: here anything from a few hundred megohms up. The general rule: choose two loads that bracket the source resistance widely.

## Your turn

1. In Part 1, change Example 06-08.3's feedback resistor to 99 kΩ. Predict whether $R_{th}$ rises or falls, then compute it.
2. In Part 2, at what load do the Thévenin and Norton internal powers become equal?
3. In Part 3, the cable is shortened to 50 m (3.125 Ω). What is the new maximum power?

<details><summary>Answers</summary>

1. It rises toward the 2 kΩ of $R_o$: a larger $R_f$ feeds back less, so there is less loop gain to lower the output resistance. With 99 kΩ the test-source equations give $v_x = v_o/100$ and $0.55v_o + 0.01v_o = 1$ mA, so $R_{th} = 1/0.56 = 1.786$ kΩ.
2. When $i_L^2R_{th} = v_L^2/R_{th}$, that is $v_L = i_LR_{th}$, so $R_L = R_{th}$ = 3 kΩ: the matched load.
3. $52^2/(4\times3.125) = 216.3$ W: halving the resistance doubles the Thévenin limit.

</details>